# 01 · Data store + evaluation setup  (steps 1-3)
**Accelerator: None (CPU).** Run with **Save Version → Save & Run All (Commit)**; you can close the laptop. Re-running continues where the last run stopped, on any account.


* builds the BSDS500 array store (agreement maps from the per-annotator ground truth) and uploads it to HF
* copies the code to HF so other accounts don't need the Kaggle code dataset
* **step 1 gate:** scores the released PEdger checkpoint in every inference mode; one of them should
  reproduce the published ODS within ±0.002 (.821/.840 BSDS+VOC or .813/.834 BSDS-only)

In [ ]:
# ==== EDIT ONLY THIS CELL (keep the same values on every Kaggle account) ====
HF_REPO = "sujalprasad90/pedger-mtkd-store"   # private Hugging Face dataset repo; created automatically
ACCOUNT = "account-1"                         # any label; recorded in progress.json (who trained what)
REPO_URL = ""                                 # optional GitHub URL of the code ("" = Kaggle dataset or HF copy)
# Kaggle setup (once per account): Add-ons -> Secrets -> add HF_TOKEN (a WRITE token) and tick it for this
# notebook; Settings -> Internet ON; Settings -> Accelerator as noted in the title.

In [ ]:
# ---- setup: secrets, code, dependencies (no edits needed) ----
import glob, os, shutil, subprocess, sys, time
SESSION_START = time.time()
from kaggle_secrets import UserSecretsClient
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
os.environ["PEDGER_REMOTE"] = "hf:" + HF_REPO
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "huggingface_hub>=0.25", "pyyaml"], check=True)
W, REPO, WS = "/kaggle/working", "/kaggle/working/pedger-mtkd", "/kaggle/working/ws"
if not os.path.exists(REPO):
    attached = sorted(glob.glob("/kaggle/input/*/pedger_kd/__init__.py") + glob.glob("/kaggle/input/*/*/pedger_kd/__init__.py"))
    if attached:                                   # code uploaded as a Kaggle Dataset (pedger-mtkd-code)
        shutil.copytree(os.path.dirname(os.path.dirname(attached[0])), REPO)
        CODE_SOURCE = "kaggle dataset"
    elif REPO_URL:
        subprocess.run(["git", "clone", "-q", REPO_URL, REPO], check=True)
        CODE_SOURCE = "git"
    else:                                          # copy pushed to Hugging Face by notebook 01
        from huggingface_hub import snapshot_download
        snapshot_download(HF_REPO, repo_type="dataset", allow_patterns=["code/*"], local_dir=f"{W}/hf_code",
                          token=os.environ["HF_TOKEN"])
        shutil.copytree(f"{W}/hf_code/code", REPO)
        CODE_SOURCE = "huggingface"
    print("code from:", CODE_SOURCE)
os.chdir(REPO)
sys.path.insert(0, REPO)
!python tools/fetch_upstream.py

In [ ]:
!pip install -q -r requirements.txt   # includes pyEdgeEval (benchmark), builds in ~1-2 min

In [ ]:
# make the code available to every account through Hugging Face
!python -m pedger_kd.sync push-code

In [ ]:
# BSDS500 original release (per-annotator ground truth); ~70 MB
RAW_BSDS = "/kaggle/working/BSR/BSDS500/data"
if not os.path.exists(RAW_BSDS):
    !wget -q https://www2.eecs.berkeley.edu/Research/Projects/CS/vision/grouping/BSR/BSR_bsds500.tgz -O /kaggle/working/bsr.tgz
    !tar -xzf /kaggle/working/bsr.tgz -C /kaggle/working && rm /kaggle/working/bsr.tgz
print(sorted(os.listdir(RAW_BSDS + "/images")))

In [ ]:
from pedger_kd.sync import open_remote
remote = open_remote(os.environ["PEDGER_REMOTE"])
if remote.exists("data/bsds/meta.json"):
    print("data/bsds already on HF, pulling it")
    remote.download("data/bsds", WS)
else:
    !python data/prepare_bsds.py --bsds-root {RAW_BSDS} --out {WS}/data/bsds
    !python -m pedger_kd.sync push-dir --local {WS}/data/bsds --to data/bsds
!cat {WS}/data/bsds/meta.json

In [ ]:
# step 3 visual check: image | annotator fraction a | label Y | trust alpha
!python data/visualize_samples.py --store {WS}/data/bsds --split train --n 10 --out {W}/checks/agreement.png
from IPython.display import Image, display; display(Image(f"{W}/checks/agreement.png"))

In [ ]:
# step 1: released PEdger checkpoint, every inference mode, SS and MS
CKPT = "third_party/PEdger/checkpoint_PEdger.pth"
for mode in ["ns_m", "rec_m", "fuse_m"]:
    for ms in ["", "--ms"]:
        out = f"{W}/released/{mode}{'_ms' if ms else ''}"
        !python eval/run_bsds_eval.py --ckpt {CKPT} --mode {mode} {ms} --store {WS}/data/bsds --split test --raw-root {RAW_BSDS} --out-dir {out} --nproc 4 --device cpu
import json
rows = []
for f in sorted(glob.glob(f"{W}/released/*/scores_*.json")):
    if f.endswith("_per_image.json"): continue
    r = json.load(open(f)); rows.append(f"{f.split('/')[-2]:10s} ODS {r['ODS']:.4f}  OIS {r['OIS']:.4f}  AP {r['AP']:.4f}")
open(f"{W}/released_scores.txt", "w").write("\n".join(rows) + "\n")
print("\n".join(rows)); print("published: .821/.840 (BSDS+VOC)   .813/.834 (BSDS only)")
!python -m pedger_kd.sync push-dir --local {W}/released --to results/released

In [ ]:
# step 9 part 1: numeric BatchNorm check on the released weights
!python eval/bn_check.py --numeric --ckpt third_party/PEdger/checkpoint_PEdger.pth --device cpu

In [ ]:
# keep the notebook output small (everything important is already on Hugging Face)
for d in ["ws/data", "ws/teachers", "ws/.staging", "hf_code", "work", "BSR"]:
    shutil.rmtree(f"/kaggle/working/{d}", ignore_errors=True)
print(f"session used {(time.time() - SESSION_START) / 3600:.2f} h")